In [60]:
import pandas as pd
import sqlite3

In [61]:
# Conectamos con la base de datos
connection = sqlite3.connect("./data/sql-murder-mystery.db")
# Obtenemos un cursor que utilizaremos para hacer las queries
cursor_mystery = connection.cursor()

In [62]:
def ejecutar_consulta(query):
    cursor_mystery.execute(query)
    resultado = cursor_mystery.fetchall()
    df = pd.DataFrame(resultado)
    display(df)
    return resultado

In [63]:
# Paso 1: Obtengo el informe de tipo "muder" ocurrido el 15/01/2018 en SQL City
ejecutar_consulta('''
SELECT description 
FROM CRIME_SCENE_REPORT 
WHERE DATE = '20180115' AND TYPE = 'murder' AND CITY = 'SQL City' 
''')


,0
0,Security footage shows that there were 2 witne...


[('Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".',)]

### El resultado anterior indica lo siguiente:
* Las imágenes de seguridad muestran que hubo dos testigos. 
  * El primer testigo vive en la última casa de "Northwestern Dr". 
  * La segunda testigo, llamada Annabel, vive en algún punto de "Franklin Ave".

### El siguiente paso será obtener de la tabla PERSON:
* La persona que vive en el número mas alto de "Northwestern Dr".
* La persona que vive en "Franklin Ave" llamada Annabel

In [64]:

# La persona que vive en el número mas alto de "Northwestern Dr"
resultado = ejecutar_consulta('''
SELECT *, MAX(ADDRESS_NUMBER) 
FROM PERSON 
WHERE ADDRESS_STREET_NAME = 'Northwestern Dr'
''')

id_testigo_1 = resultado[0][0]
print(id_testigo_1)

# La persona que vive en "Franklin Ave" llamada Annabel
resultado = ejecutar_consulta('''
SELECT * FROM PERSON WHERE ADDRESS_STREET_NAME = 'Franklin Ave' AND NAME like '%Annabel%'
''')
id_testigo_2 = resultado[0][0]
print(id_testigo_2)


,0,1,2,3,4,5,6
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949,4919


14887


,0,1,2,3,4,5
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


16371


In [65]:
# Vamos a revisar las entrevistas de estos dos sugetos para ver que han dicho

resultado = ejecutar_consulta(f'''
SELECT * 
FROM INTERVIEW
WHERE PERSON_ID IN ({id_testigo_1},  {id_testigo_2})

'''
)
for r in resultado:
    print(r[1])



,0,1
0,14887,I heard a gunshot and then saw a man run out. ...
1,16371,"I saw the murder happen, and I recognized the ..."


I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".
I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


Del primer testigo podemos afirmar:
* El número de afiliación del gym comienza por "48Z"
* Es miembro oro 
* El hombre se subió a un coche con una matrícula que incluía "H42W"

Del segundo testigo podemos afirmar:
* Estuvo en el gym el 9 de Enero

In [66]:
# Busco los miembros del gym cuyo número comienza por el indicado, que sean gold y que hayan estado en el gym el 09 de Enero.
# Busco además si tiene un conche que contenga parte de la matricula indicada.
ejecutar_consulta('''
SELECT P.NAME FROM GET_FIT_NOW_MEMBER AS M
INNER JOIN GET_FIT_NOW_CHECK_IN C
ON M.ID = C.MEMBERSHIP_ID
INNER JOIN PERSON AS P
ON P.ID = M.PERSON_ID
INNER JOIN DRIVERS_LICENSE D
ON D.ID = P.LICENSE_ID
WHERE M.ID LIKE '48Z%' AND M.MEMBERSHIP_STATUS = 'gold' AND C.CHECK_IN_DATE = '20180109' AND D.PLATE_NUMBER LIKE '%H42W%'
'''
)

,0
0,Jeremy Bowers


[('Jeremy Bowers',)]

In [67]:
# Cerramos las conexiones
cursor_mystery.close()
connection.close()

# **El asesino es Jeremy Bowers**